# Session 10 — Modules, OOP & the Pythonic Toolkit

> import modules; a class with @property; generators/map/filter/walrus.

**How to use this notebook:** read each cell, **predict** what it prints, then run it with **Shift + Enter**. Change one thing and predict again — the surprise is the lesson. Practice tasks (with collapsed solutions) are at the bottom.

In [ ]:
# Setup (notebook only): write the module this session imports, then make it importable.
import sys
from pathlib import Path
Path('grades.py').write_text('"""grades.py — a small reusable module. Import it: `from grades import letter_grade`."""\n\n\ndef letter_grade(score: float) -> str:\n    """Return A/B/C/D/F by 90/80/70/60 cutoffs."""\n    for cutoff, letter in [(90, "A"), (80, "B"), (70, "C"), (60, "D")]:\n        if score >= cutoff:\n            return letter\n    return "F"\n\n\ndef class_average(scores: list[float]) -> float:\n    """Return the mean of scores."""\n    return sum(scores) / len(scores)\n\n\nif __name__ == "__main__":\n    # Runs only when you execute `python3 grades.py`, not when imported.\n    print(letter_grade(85), class_average([91, 58, 73]))\n')
sys.path.insert(0, '.')
print('wrote grades.py')

In [ ]:
import functools
import itertools
from dataclasses import dataclass

from grades import letter_grade, class_average   # import from our own module (grades.py)

In [ ]:
# --- 1. Modules: reuse code from another file ---------------------------
print("letter for 85:", letter_grade(85))
print("average:", class_average([91, 58, 73]))

In [ ]:
# --- 2. OOP: model a domain entity --------------------------------------
class Student:
    def __init__(self, name: str, gpa: float):
        self.name = name
        self.gpa = gpa             # runs through the setter below (validates!)

    def __str__(self) -> str:      # how the object prints
        return f"{self.name}: {self.gpa} ({self.standing()})"

    def standing(self) -> str:
        return "Good" if self.gpa >= 2.0 else "Probation"

    @property                       # makes .gpa look like an attribute...
    def gpa(self) -> float:
        return self._gpa

    @gpa.setter                     # ...but validates on every assignment
    def gpa(self, value: float):
        if not 0 <= value <= 4:
            raise ValueError(f"gpa {value} not in 0-4")
        self._gpa = value

    @classmethod                    # an alternate constructor: build from a CSV row
    def from_row(cls, row: str) -> "Student":
        name, gpa = row.split(",")
        return cls(name.strip(), float(gpa))

    @staticmethod                   # a helper that needs no instance/class
    def is_passing(gpa: float) -> bool:
        return gpa >= 2.0


ana = Student("Ana", 3.9)
print("\n", ana)                    # uses __str__
print("from_row:", Student.from_row("Ben, 3.4"))     # classmethod constructor
print("is_passing(1.5)?", Student.is_passing(1.5))   # staticmethod
try:
    ana.gpa = 5.0                   # rejected by the setter
except ValueError as e:
    print("rejected:", e)

In [ ]:
# --- 3. Inheritance ------------------------------------------------------
class GradStudent(Student):
    def __init__(self, name, gpa, advisor):
        super().__init__(name, gpa)     # reuse the parent's setup
        self.advisor = advisor
    def __str__(self):
        return super().__str__() + f" — advised by {self.advisor}"

print("\n", GradStudent("Cara", 3.4, "Dr. Lee"))

In [ ]:
# --- 4. @dataclass: __init__, __repr__, __eq__ written for you ----------
@dataclass
class Grade:
    course: str
    score: int

g1, g2 = Grade("ED1234", 91), Grade("ED1234", 91)
print("\ndataclass repr:", g1)          # Grade(course='ED1234', score=91) — free __repr__
print("value equality:", g1 == g2)      # True — free __eq__ compares fields

In [ ]:
# --- 5. match / case: structural pattern matching (Python 3.10+) --------
def classify(record):
    match record:
        case {"gpa": g} if g >= 3.5:        # mapping pattern + guard
            return "honors"
        case {"gpa": _}:
            return "regular"
        case [first, *_]:                   # sequence pattern, capture the head
            return f"list starting with {first!r}"
        case _:                             # wildcard — the default
            return "unknown"

print("\nclassify:", classify({"gpa": 3.9}), "|", classify([10, 20]), "|", classify(42))

In [ ]:
# --- 6. The Pythonic toolkit --------------------------------------------
roster = [Student("Ana", 3.9), Student("Ben", 1.8), Student("Cara", 3.2)]

print("\ngood standing:", [s.name for s in roster if s.gpa >= 2.0])   # comprehension
print("upper:", list(map(lambda s: s.name.upper(), roster)))         # map
at_risk = filter(lambda s: s.gpa < 2.0, roster)                      # filter the objects
print("at risk:", [s.name for s in at_risk])
print("all passing?", all(s.gpa >= 2.0 for s in roster))            # any/all over a generator

# reduce: fold a sequence down to a single value (here: product of gpas)
print("gpa product:", round(functools.reduce(lambda a, s: a * s.gpa, roster, 1.0), 2))

# itertools.groupby: group ADJACENT items — so sort by the key first!
by_standing = sorted(roster, key=lambda s: s.standing())
for standing, group in itertools.groupby(by_standing, key=lambda s: s.standing()):
    print(f"  {standing}: {[s.name for s in group]}")

# generator: produce values lazily; great for huge data
def gpas(students):
    for s in students:
        yield s.gpa

gen = gpas(roster)
print("\nmean gpa:", round(class_average(list(gen)), 2))
print("second pass:", list(gen))   # [] — a generator is exhausted after one pass

# walrus := : assign and test in one step
if (n := len(roster)) > 2:
    print(f"\n{n} students enrolled")

## Now you try — practice

# Session 10 — Practice (60 min): Modules, OOP & Pythonic

Files in this folder: `grades.py` (a module you import), `demo.py` (worked example).

## Task 1 — Use a module
From a new file, `from grades import letter_grade, class_average` and call both. Why does the
`if __name__ == "__main__":` block in `grades.py` NOT run when you import it?

## Task 2 — A class with a validating property
Build `Student(name, gpa)` with:
- `__str__` → `"Ana: 3.9 (Good)"`,
- `standing()` → `"Good"` if gpa ≥ 2.0 else `"Probation"`,
- a `@property` setter for `gpa` that raises `ValueError` outside 0–4.
Prove the setter rejects `5.0`.

## Task 3 — Inheritance
Add `GradStudent(Student)` that also stores an `advisor` and uses `super().__init__(...)`.
Override `__str__` to append the advisor.

## Task 4 — The Pythonic toolkit
Given a roster of `Student`s:
1. names in good standing (list comprehension),
2. uppercase names (`map`),
3. at-risk students (`filter`),
4. mean gpa via a **generator** that `yield`s each gpa — then show the generator is empty on a
   second pass.

## Bonus — Pythonic idiom drill
Cover the `# ->` answers, predict each line, then run.

```python
from dataclasses import dataclass

@dataclass                           # auto __init__, __repr__, __eq__
class Point:
    x: int
    y: int
print(Point(1, 2), Point(1, 2) == Point(1, 2))   # -> Point(x=1, y=2) True

def head(v):
    match v:                         # structural pattern matching (3.10+)
        case [first, *_]: return first
        case _: return None
print(head([9, 8]), head(5))         # -> 9 None
```

---

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show solutions</strong></summary>

See `demo.py` — it implements Tasks 2–4. Key points:

```python
@property
def gpa(self): return self._gpa
@gpa.setter
def gpa(self, v):
    if not 0 <= v <= 4: raise ValueError(...)
    self._gpa = v

class GradStudent(Student):
    def __init__(self, name, gpa, advisor):
        super().__init__(name, gpa)
        self.advisor = advisor

[s.name for s in roster if s.gpa >= 2.0]          # comprehension
list(map(lambda s: s.name.upper(), roster))        # map
[s.name for s in filter(lambda s: s.gpa < 2.0, roster)]   # filter
def gpas(rs):
    for s in rs: yield s.gpa                        # generator (exhausts after one pass)
```
Task 1: the `__name__` guard is only `"__main__"` when the file is **run directly**; on `import`
its `__name__` is `"grades"`, so the demo block is skipped — that's how a file can be both a
runnable script and an importable module.

</details>